# Lekcja 7 — Neural Receiver (CNN → LLR → LDPC)

## Cel
Zastąpić trzy bloki klasyczne **jednym CNN**:
```
RX Grid [B, 2, T, F] → CNN → LLR → LDPC → Bits
```

## Architektura (uproszczona)
```
Conv2D(2→32) → ReLU → Conv2D → ReLU → Conv2D → GAP → Linear → LLR
```

## Loss
Nie BLER (niedifferentiowalny), tylko **Binary Cross-Entropy** na LLR:
$$ \mathcal{L} = \text{BCE}(\text{bits}, \text{LLR}) $$

## Co NIE jest neuronowe
- **Transmitter** (LDPC, mapper, grid) — generuje dane treningowe
- **LDPC decoder** — zostaje klasyczny


In [ ]:
import sys
from pathlib import Path

# Dodaj src/ do PYTHONPATH
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch

try:
    import sionna as sn
    import sionna.phy
except ImportError as e:
    raise ImportError(
        "Brak Sionny. Uruchom z katalogu magisterka/: ./scripts/drun sync"
    ) from e

from src.utils.setup import print_environment, get_device

sn.phy.config.seed = 42
device = get_device()
print_environment()


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from src.models.neural_receiver import NeuralReceiverCNN

# Przykładowe wymiary (dopasuj do swojego resource grid)
BATCH = 8
T, F = 14, 64
NUM_BITS = 512

model = NeuralReceiverCNN(num_bits=NUM_BITS, hidden_channels=32).to(device)
y_fake = torch.randn(BATCH, 2, T, F, device=device)
llr = model(y_fake)
print("LLR shape:", llr.shape)


## Trening (szkielet pętli)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# W prawdziwym treningu: generujesz (y_grid, bits) z pipeline Sionna
bits_train = torch.randint(0, 2, (BATCH, NUM_BITS), device=device).float()
llr_out = model(y_fake)

loss = F.binary_cross_entropy_with_logits(llr_out, bits_train)
loss.backward()
optimizer.step()
optimizer.zero_grad()
print("Loss po 1 kroku:", loss.item())


## Porównanie z baseline

In [ ]:
# Po wytrenowaniu: ta sama pętla SNR co w lekcji 06
# neural_bler vs classical_bler na jednym wykresie

plt.figure(figsize=(8, 5))
plt.semilogy([0, 2, 4, 6, 8, 10], [0.5, 0.3, 0.15, 0.05, 0.02, 0.01], "--", label="Classical (przykład)")
plt.semilogy([0, 2, 4, 6, 8, 10], [0.45, 0.25, 0.10, 0.03, 0.01, 0.005], "-", label="CNN (docelowo Twój wynik)")
plt.xlabel("Eb/N0 [dB]")
plt.ylabel("BLER")
plt.legend()
plt.title("Classical vs Neural — cel pracy magisterskiej")
plt.grid(True, which="both")
plt.show()


## Podsumowanie całej ścieżki nauki

| Lekcja | Zrozumiałeś | Zastąpione przez NN |
|--------|-------------|---------------------|
| 01 QPSK+AWGN | bity, LLR, BER | — |
| 02 OFDM | resource grid | wejście CNN |
| 03 Fading | H(f), piloty | estymacja H |
| 04 MIMO | wymiary tensora | detekcja MIMO |
| 05 PUSCH | LDPC 5G | — (decoder zostaje) |
| 06 Classical | LS + LMMSE + demap | **cały ten łańcuch** |
| 07 Neural | CNN → LLR | trening + ewaluacja |

## Kolejne kroki (poza lekcjami)
1. Pełny trening CNN na OFDM+TDL
2. Generalization (train CDL-A, test CDL-B/C)
3. Latency / FLOPs / deployment (ONNX, TensorRT)

Gratulacje — masz mapę od zera do Neural Receiver!
